In [4]:
library(ggplot2)
library(dplyr)
library(readr) # 用于读取 csv

# ==============================================================================
# 1. 定义绘图函数 (核心部分)
# ==============================================================================
plot_selected_h2 <- function(file_path, target_phenotypes) {
  
  # --- A. 读取数据 ---
  # 假设你的文件第一列是表型名称，表头可能是 "Unnamed: 0" 或其他
  # 我们读取后强制重命名第一列为 "Phenotype"
  df <- read_csv(file_path, show_col_types = FALSE)
  colnames(df)[1] <- "Phenotype" # 修复列名
  
  # --- B. 筛选数据 ---
  # 核心：只保留在 target_phenotypes 列表中的行
  plot_data <- df %>%
    filter(Phenotype %in% target_phenotypes) %>%
    # 按遗传力排序 (为了让棒棒糖按高低排列)
    arrange(H2) %>%
    # 将表型转为因子并固定顺序 (否则 ggplot 会默认按字母排序)
    mutate(Phenotype = factor(Phenotype, levels = Phenotype))
  
  # 检查是否筛选到了数据
  if(nrow(plot_data) == 0) {
    stop("错误: 没有找到匹配的表型，请检查你的拼写是否正确！")
  }
  print(paste("成功筛选出", nrow(plot_data), "个表型进行绘图"))
  
  # --- C. 绘图 ---
  p <- ggplot(plot_data, aes(x = Phenotype, y = H2)) +
    # 1. 画棒 (Segment)
    geom_segment(aes(x = Phenotype, xend = Phenotype, y = 0, yend = H2), 
                 color = "grey70", linewidth = 1) +
    
    # 2. 画糖 (Point)
    # 颜色映射到 H2 数值，大小设为 5
    geom_point(aes(color = H2), size = 5) +
    
    # 3. 样式调整
    coord_flip() + # 翻转坐标轴，变成横向
    scale_color_viridis_c(option = "D", name = "Heritability", limits = c(0, 1)) +
    scale_y_continuous(limits = c(0, 1), expand = c(0, 0.05)) + # Y轴范围 0-1
    
    theme_classic(base_size = 14) + # 学术风格主题
    theme(
      axis.text.y = element_text(color = "black", face = "bold"),
      panel.grid.major.y = element_line(color = "grey90", linetype = "dashed"), # 加辅助线
      legend.position = "right"
    ) +
    
    labs(
         x = "", # 去掉表型轴的标题，因为标签已经很明显了
         y = expression(bold(Heritability~(h^2))))
  
  # --- D. 保存与显示 ---
  ggsave("Selected_Phenotype_Heritability.pdf", p, width = 8, height = 6)
  print(p)
  print("图片已保存为: Selected_Phenotype_Heritability.pdf")
}

# ==============================================================================
# 2. 用户使用区 (在这里修改)
# ==============================================================================

# 你的文件路径
my_file <- "./data/all_phenotype_h2.csv"

# 【关键】在这里输入你想要画的表型列表
# 只有在这个列表里的才会显示
# my_target_list <- c(
#   'RGBVI', 'WI', 'ExB', 'COM',
#     'RGB_B', 'GreenCoverage', 'ints', 'veg', 'bn', 'RGB_G',
#     'RGB_R', 'IPCA', 'GRVI', 'NDI', 'MGRVI',
#     'IKAW', 'GLI2', 'rn'
# )

my_target_list <- c(
  'RGBVI', 'WI', 'ExB', 'COM',
    'GreenCoverage',  'bn',
    'IPCA', 'NDI'
)

# 调用函数
plot_selected_h2(my_file, my_target_list)

New names:
• `` -> `...1`


[1] "成功筛选出 8 个表型进行绘图"


Warning message in grid.newpage():
“Cannot open temporary file '/tmp/Rtmp2eigtJ/pdf1f9d57346c622' for compression (reason: 没有那个文件或目录); compression has been turned off for this device”


[1] "图片已保存为: Selected_Phenotype_Heritability.pdf"


ERROR: Error in value[[3L]](cond): could not open file '/tmp/Rtmp2eigtJ/file1f9d557aa87b.png'


plot without title